# Lenguajes y Autómatas II (SCD-1016)
## Sesión 11 · Arquitectura de la Tabla de Símbolos

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Dónde guarda el compilador el nombre de cada variable mientras lee el programa?
>
> **Objetivo de la sesión:** Entender por qué la tabla de símbolos usa una función de dispersión y cadenas de colisión, e implementar una tabla que inserte nombres sin perder información cuando dos claves caen en la misma cubeta.

**Entregable:** tus funciones hash_djb2() e insertar_simbolo() en artesano_sesion11_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion11.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un arreglo de cubetas vacías y una secuencia de nombres: calcula la posición con djb2, crea un nodo por símbolo nuevo y recorre la cadena cuando dos nombres coinciden en la misma cubeta. insertar_simbolo() devuelve 1 si insertó, 0 si el nombre ya existía y -1 ante argumentos inválidos; el harness comprueba cuántos nodos hay y qué offset conserva cada uno.

Al final de la sesión tendrás: **Una tabla hash en C que inserta símbolos, rechaza nombres repetidos y conserva las colisiones en cadenas enlazadas, junto con un harness en verde que lo demuestra.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 11](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion11.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion11.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion11.h"

unsigned int hash_djb2(const char *clave);
int insertar_simbolo(Simbolo *tabla[], unsigned int cubetas, const char *nombre, int tipo, int offset);
```

- Ningún nombre se duplica: si ya existe en cualquier cubeta, insertar devuelve 0 y no crea nodo.
- Una colisión se encadena; jamás se sobrescribe una entrada existente.
- Con tabla NULL, nombre NULL o cero cubetas, insertar devuelve -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion11_c.c` junto con `artesano_sesion11_c.c`
> (o `ia_sesion11_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `hash_djb2("")` | 5381 |
| 2 | `hash_djb2("abc")` | 193485963 |
| 3 | `insertar x dos veces` | 0 |
| 4 | `dos claves, una cubeta` | 2 |
| 5 | `offset de beta en cadena` | 4 |
| 6 | `insertar en tabla NULL` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 11 - Tabla hash con encadenamiento ===
Caso                               esperado:              obtenido:              resultado
[ 1] hash_djb2("")                      esperado: 5381         obtenido: 5381         OK
[ 2] hash_djb2("abc")                   esperado: 193485963    obtenido: 193485963    OK
[ 3] insertar x dos veces               esperado: 0            obtenido: 0            OK
[ 4] dos claves, una cubeta             esperado: 2            obtenido: 2            OK
[ 5] offset de beta en cadena           esperado: 4            obtenido: 4            OK
[ 6] insertar en tabla NULL             esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion11.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] hash_djb2() reproduce la fórmula djb2 para claves vacías, cortas y repetidas.
- [ ] Dos nombres con el mismo índice quedan en la misma cadena y ninguno se pierde.
- [ ] El harness reporta 6/6 en verde y el esqueleto inicial falla antes de implementar.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion11_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion11.h
/* api_sesion11.h - contrato de la Sesion 11
 * Tabla hash con encadenamiento
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION11_H
#define API_SESION11_H

typedef struct Simbolo {
    char nombre[16];
    int tipo;
    int offset;
    struct Simbolo *sig;
} Simbolo;

unsigned int hash_djb2(const char *clave);
int insertar_simbolo(Simbolo *tabla[], unsigned int cubetas, const char *nombre, int tipo, int offset);

#endif /* API_SESION11_H */


In [ ]:
%%writefile artesano_sesion11_c.c
/* artesano_sesion11_c.c - Fase 1: El Artesano (sin IA)
 * Tabla hash con encadenamiento.
 *
 * Implementa lo que declara api_sesion11.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion11.h"
#include <stdio.h>
#include <string.h>

unsigned int hash_djb2(const char *clave) {
    (void)clave; /* TODO: empieza en 5381 y multiplica por 33 en cada byte */
    return 0;
}

int insertar_simbolo(Simbolo *tabla[], unsigned int cubetas, const char *nombre, int tipo, int offset) {
    (void)tabla; (void)cubetas; (void)nombre; (void)tipo; (void)offset; /* TODO */
    return -1;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion11_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion11_c.c
/* ia_sesion11_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion11.h. No puede traer main().
 */
#include "api_sesion11.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion11_c.c
/* harness_sesion11_c.c - auditoria automatica de la Sesion 11
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion11_c.c artesano_sesion11_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion11.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 11 - Tabla hash con encadenamiento ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { unsigned int r = hash_djb2(""); verificar_int("hash_djb2(\"\")", (long)(5381), (long)(r)); }
    { unsigned int r = hash_djb2("abc"); verificar_int("hash_djb2(\"abc\")", (long)(193485963), (long)(r)); }
    { Simbolo *t[8] = {0}; insertar_simbolo(t, 8, "x", 1, 0); int r = insertar_simbolo(t, 8, "x", 2, 8); verificar_int("insertar x dos veces", (long)(0), (long)(r)); }
    { Simbolo *t[1] = {0}; insertar_simbolo(t, 1, "alpha", 1, 0); insertar_simbolo(t, 1, "beta", 2, 4); int r = 0; for (Simbolo *p = t[0]; p != NULL; p = p->sig) r++; verificar_int("dos claves, una cubeta", (long)(2), (long)(r)); }
    { Simbolo *t[1] = {0}; insertar_simbolo(t, 1, "alpha", 1, 0); insertar_simbolo(t, 1, "beta", 2, 4); int r = -1; for (Simbolo *p = t[0]; p != NULL; p = p->sig) { if (strcmp(p->nombre, "beta") == 0) r = p->offset; } verificar_int("offset de beta en cadena", (long)(4), (long)(r)); }
    { int r = insertar_simbolo(NULL, 8, "x", 1, 0); verificar_int("insertar en tabla NULL", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion11.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion11_c.c artesano_sesion11_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion11_c.c ia_sesion11_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion11.md` a `auditorias/auditoria_sesion11.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
